# Week 9 (CS 418 @ UIC)

<!-- slide 1 -->
# Week 9 Slide Deck

<h2>Decision Trees and KNNs; SVMs and Kernels</h2>

Jack Bandy
2026

<!-- slide 2 -->
# Week 9, Day 1

CS 418 · Week 9 · 🟠 Halsted 🟠

<!-- slide 3 -->
## Map for Week 9, Day 1

- Data From Neighbors
- Similarity Metrics
- Distance Metrics
- k-NN Details
- Decision Tree Examples
- Decision Tree Details

![](../assets/orange-line-map-stops/stop10-halsted.svg)

<!-- slide 4 -->
## Any questions?

- PrairieLearn Homeworks?
- Group Projects?
- Week 9 Quiz is Wednesday, in class
- Other logistics?
- (You can also stay after class for questions)

<!-- slide 5 -->
# Collect Data From Neighbors

<!-- slide 6 -->
## The Self

<!-- slide 7 -->
## Find Your Neighbors

Worksheet checkpoint 1:

- Fill in the **Self** column: 1 if you have taken the course, 0 if not.
- Talk to five classmates and fill in a column for each one.
    - Only one of the five can be in your project group.

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

<!-- slide 8 -->
## Predict Alice From Her Classmates

Source: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 3](http://ciml.info/dl/v0_99/ciml-v0_99-ch03.pdf) ("Geometry and Nearest Neighbors").

suppose you need to predict whether Alice will like [a class]. Perhaps we can try to find another student who is most 'similar' to Alice, in terms of favorite courses.

- Our version: did Alice take **CS 412** (Intro to Machine Learning)?
    - **predictors $X$**: the other 11 courses, took it (1) or not (0)
    - **class label $y$**: took CS 412 (▲) or not (●)
- "If it walks like a duck, quacks like a duck, and looks like a duck..."
- A $k$-nearest neighbors classifier assigns class labels to observations based on the class labels of the $k$ 'most similar' observations

<!-- slide 9 -->
## Synthetic Class Survey

<img src="../assets/week9/survey-grid-v0.svg" class="figure figure-plain" style="max-height:440px" alt="A grid like worksheet checkpoint 1: twelve course rows, CS 100 through CS 488, and twenty classmate columns. A filled square means the classmate took the course. Every classmate took CS 251.">

<img src="../assets/week9/survey-grid-v1.svg" class="figure figure-plain" style="max-height:440px" alt="The CS 412 row moves below a dashed line and becomes the label: six orange triangles for classmates who took CS 412 and fourteen gray circles for those who did not.">

<img src="../assets/week9/survey-grid-v2.svg" class="figure figure-plain" style="max-height:440px" alt="A new column for Alice, outlined: she took CS 251, CS 401, CS 411, and CS 480. Her CS 412 cell is a question mark.">

Synthetic: 20 classmates and Alice, from the code demo's generator.

<!-- slide 10 -->
## Each Student as a Vector

Source: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 3](http://ciml.info/dl/v0_99/ciml-v0_99-ch03.pdf), §3.1 ("From Data to Feature Vectors").

- Obviously a student is much more than a vector
- Example: Alice $= (0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0)$
    - one entry per course

- prediction tasks as mapping inputs to outputs
    - each input is "nothing more than lists of feature values." (CIML)
- Binary features become 0 (for false) or 1 (for true)
- Categorical features (say, major: CS, DS, or CE)
    - map to V-many binary indicator features.
    - Why not CS = 0, DS = 1, CE = 2?
    - That model suggests that CS is closer to DS than to CE

<!-- slide 11 -->
# Similarity Metrics

<!-- slide 12 -->
## Similarity Metrics for Binary Features

$N_{11}$: attributes where both $a$ and $b$ are 1; $N_{00}$: both 0; $N_{10}$, $N_{01}$: they disagree.

- **Jaccard**: $S(a, b) = \dfrac{N_{11}}{N_{01} + N_{10} + N_{11}}$
    - (or [`sklearn.metrics.jaccard_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.jaccard_score.html) 🙂)
- **Simple matching (SMC)**: $S(a, b) = \dfrac{N_{00} + N_{11}}{N_{00} + N_{01} + N_{10} + N_{11}}$
    - (or 1 − [`scipy.spatial.distance.hamming`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.hamming.html) 🙂)

<img src="../assets/week9/survey-pair-v2.svg" class="figure figure-plain" style="max-height:260px" alt="Alice and classmate number 5 with each course labeled by type: N11 for CS 251, 401, 411, and 480; N01 for CS 301, 361, and 424; N00 for the other four. N11 is 4, N10 is 0, N01 is 3, N00 is 4. Jaccard is 4 out of 7, about 0.571; SMC is 8 out of 11, about 0.727.">

- difference: does a shared 0 count as agreement?

<!-- slide 13 -->
## Who Is Alice's Nearest Neighbor?

<img src="../assets/week9/survey-neighbors-v0.svg" class="figure figure-plain" style="max-height:440px" alt="All twenty classmates ranked by Jaccard similarity to Alice. Number 5 is first at 0.571, then numbers 3 and 6 tied at 0.500.">

<img src="../assets/week9/survey-neighbors-v1.svg" class="figure figure-plain" style="max-height:440px" alt="Added on the right: the same classmates ranked by SMC. Number 6 is first at 0.818, ahead of a six-way tie at 0.727 that includes number 5.">

#6 took only CS 251 and CS 480, so they match Alice mostly on shared 0s, which SMC counts and Jaccard ignores.

<!-- slide 14 -->
# Distance Metrics

<!-- slide 15 -->
## Distance Metrics

Used to measure the dissimilarity between two observations. For continuous attributes, with $p$ attributes:

- **Euclidean**: $d(a, b) = \sqrt{\sum_{i=1}^{p} (a_i - b_i)^2}$
    - (or [`sklearn.metrics.pairwise.euclidean_distances`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.euclidean_distances.html) 🙂)
- **Manhattan**: $d(a, b) = \sum_{i=1}^{p} |a_i - b_i|$
    - (or [`sklearn.metrics.pairwise.manhattan_distances`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.manhattan_distances.html) 🙂)

<!-- slide 16 -->
## Worked Example: $a = (3, 4)$ and $b = (2, 2)$

Source: Gonzalo Bello Lander, CS 418 Lecture 13, "Classification (II)" (2026), slide 5.

**Euclidean** (straight line)

$\sqrt{(3-2)^2 + (4-2)^2} = \sqrt{1 + 4} = \sqrt{5} \approx 2.24$

**Manhattan** (city blocks)

$|3 - 2| + |4 - 2| = 1 + 2 = 3$

<img src="../assets/week9/distance-v0.svg" class="figure figure-plain" style="max-height:400px" alt="Points a at (3, 4) and b at (2, 2) on a grid.">

<img src="../assets/week9/distance-v1.svg" class="figure figure-plain" style="max-height:400px" alt="An orange straight line joins b to a, labeled Euclidean, square root of 5, about 2.24.">

<img src="../assets/week9/distance-v2.svg" class="figure figure-plain" style="max-height:400px" alt="Added: a dashed gray path from b one unit right then two units up to a, labeled Manhattan, 1 plus 2 equals 3.">

<!-- slide 17 -->
## Distance on 0/1 Data

<img src="../assets/week9/survey-pair-v0.svg" class="figure figure-plain" style="max-height:330px" alt="Alice and classmate number 5 side by side, one column per course from CS 100 to CS 488. Filled squares mark courses taken.">

<img src="../assets/week9/survey-pair-v1.svg" class="figure figure-plain" style="max-height:330px" alt="Orange boxes mark the three courses where they differ, CS 301, CS 361, and CS 424, all taken by number 5 but not Alice. Manhattan distance is 3; Euclidean distance is the square root of 3, about 1.73.">

- Each $|a_i - b_i|$ is 1 where they differ and 0 where they agree
    - Manhattan distance just counts the courses that differ
- Euclidean distance is the square root of that same count
    - both metrics rank classmates the same way

<!-- slide 18 -->
# k-NN Details

<!-- slide 19 -->
## The k-NN Algorithm

Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 3](http://ciml.info/dl/v0_99/ciml-v0_99-ch03.pdf), §3.2 and Algorithm 3.

1. Choose a proximity metric and a number of nearest neighbors $k$.
2. For each observation $z$ in the test set:
    1. Compute the distance or similarity between $z$ and every observation in the training set.
    2. Select the $k$ observations most similar to $z$ ($k$ nearest neighbors).
    3. Assign a class label to $z$ based on the class labels of its $k$ nearest neighbors. (majority vote)

- Note there actually is no 'training' phase for K-nearest neighbors
    - **lazy learner**.
- What about ties?
- Why is it a good idea to use an odd number for K?

<!-- slide 20 -->
## k-NN for Alice, One Step at a Time

<img src="../assets/week9/survey-knn-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Alice's seven most similar classmates by Jaccard similarity, each with their CS 412 answer: number 5, 0.571, took it; numbers 3 and 6, 0.500, did not; number 12, 0.429, took it; numbers 2, 13, and 19, 0.400, did not.">

<img src="../assets/week9/survey-knn-v1.svg" class="figure figure-plain" style="max-height:420px" alt="A box around the single nearest neighbor, number 5. k equals 1: one triangle versus zero circles, so predict triangle.">

<img src="../assets/week9/survey-knn-v2.svg" class="figure figure-plain" style="max-height:420px" alt="A dashed box around the three nearest neighbors. k equals 3: one triangle versus two circles, so predict circle.">

<img src="../assets/week9/survey-knn-v3.svg" class="figure figure-plain" style="max-height:420px" alt="Alice's real answer is revealed: circle, she did not take CS 412. The k equals 1 prediction is marked wrong and the k equals 3 prediction right.">

One neighbor "cannot consider the 'preponderance of evidence'." (CIML Fig. 3.4). What would $k = 2$ do?

<!-- slide 21 -->
## Memorizing vs. Generalizing

Source: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 1](http://ciml.info/dl/v0_99/ciml-v0_99-ch01.pdf), §1.1 ("What Does it Mean to Learn?").

Alice is taking a machine learning course, and her teacher Bob writes the exam. What makes a reasonable exam?

- **Unfair**: "If Bob spends the entire semester talking about machine learning, and then gives Alice an exam on History of Pottery, then Alice's performance on this exam will not be representative of her learning."
- **Memorization**: "if the exam only asks questions that Bob has answered exactly during lectures, then this is also a bad test of Alice's learning, especially if it's an 'open notes' exam."
- **Generalization**: "What is desired is that Alice observes specific examples from the course, and then has to answer new, but related questions on the exam. This tests whether Alice has the ability to generalize."

<!-- slide 22 -->
## Generalization

> Generalization is perhaps the most central concept in machine learning.

Hal Daumé III

Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)*, v0.99, [ch. 1](http://ciml.info/dl/v0_99/ciml-v0_99-ch01.pdf), §1.1.

<!-- slide 23 -->
## Memorizing or Generalizing?

Worksheet checkpoint 5:

- In your own words, what is the difference between memorizing and generalizing?
- With $k = 1$, every training point's nearest neighbor is itself. Is that memorizing or generalizing? Why?

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

<!-- slide 24 -->
## Choosing k: Overfitting vs. Underfitting

- with K = 1, risk of overfitting
- if K is large (for instance, K = N), then KNN-Predict will always predict the majority class (underfitting)
- K is a **hyperparameter** of the KNN algorithm
    - trade-off between overfitting (small value of K) and underfitting (large value of K).
- "a learned model with a decision boundary that is really jagged (like the coastline of Norway) is really complex and prone to overfitting."
- "A learned model with a decision boundary that is really simple (like the bounary [*sic*] between Arizona and Utah) is potentially underfit." (CIML)

<!-- slide 25 -->
## Choosing k on a Bigger Class

<img src="../assets/week9/survey-choose-k-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Validation accuracy against k on a log scale from 1 to 599: 70 percent at k equals 1, rising to the mid 80s between about 5 and 75, then falling.">

<img src="../assets/week9/survey-choose-k-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added: a dashed line at 74 percent, the accuracy of always guessing circle. At k of 301 and above the curve sits on that line.">

<img src="../assets/week9/survey-choose-k-v2.svg" class="figure figure-plain" style="max-height:420px" alt="The best point is circled: k equals 15, 86 percent. The left end is labeled small k, overfit; the right end k near N, underfit.">

The code demo's 1,000 synthetic students: 600 training, 200 validation, Jaccard similarity.

<!-- slide 26 -->
## k Is a Hyperparameter

<img src="../assets/week9/k-accuracy-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Training accuracy against k on a log scale from 1 to 149: 100 percent at k equals 1, falling to about 83 percent by k of 21, then dropping to about 56 percent near k equals N.">

<img src="../assets/week9/k-accuracy-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added in orange, held-out accuracy: about 74 percent at k equals 1, rising to about 82 percent, flat through k of 100, then dropping to about 54 percent.">

<img src="../assets/week9/k-accuracy-v2.svg" class="figure figure-plain" style="max-height:420px" alt="The peak of the held-out curve is circled, labeled best held-out: k equals 75. Left end labeled overfit, k equals 1; right end labeled underfit, k near N.">

"So, K is a hyperparameter of the KNN algorithm that allows us to trade-off between overfitting (small value of K) and underfitting (large value of K)." (CIML). Synthetic data, 10% labels flipped; average of 40 draws of 150 training and 1,500 held-out points.

<!-- slide 27 -->
## Example: Choosing k in Two Dimensions

<img src="../assets/week9/choose-k-v0.svg" class="figure figure-plain" style="max-height:440px" alt="Sixty training points, 27 triangles and 33 circles, overlapping in the middle. Three points are ringed: two circles inside the triangle region and one triangle inside the circle region.">

<img src="../assets/week9/choose-k-v1.svg" class="figure figure-plain" style="max-height:440px" alt="k equals 1: the boundary is tangled, with small islands around each of the noisy circled points.">

<img src="../assets/week9/choose-k-v2.svg" class="figure figure-plain" style="max-height:440px" alt="k equals 7: a smoother boundary with only small wiggles.">

<img src="../assets/week9/choose-k-v3.svg" class="figure figure-plain" style="max-height:440px" alt="k equals 60, all the data: the whole plane is gray, every point is predicted circle, the majority class.">

The decision boundary is "the line with positive land on one side and negative land on the other side." (CIML §3.3). Synthetic; three labels flipped on purpose.

<!-- slide 28 -->
## Pitfall: Irrelevant Features

<img src="../assets/week9/survey-month-v0.svg" class="figure figure-plain" style="max-height:440px" alt="All twenty classmates ranked by Manhattan distance to Alice, the number of courses that differ. Number 6 is nearest, with 2.">

<img src="../assets/week9/survey-month-v1.svg" class="figure figure-plain" style="max-height:440px" alt="Each bar gets an orange extension for the gap between that classmate's birth month and Alice's, month 4. Number 6, born in month 1, falls to a total of 5. Number 5, born in month 4, becomes nearest with a total of 3.">

Add a synthetic birth month (1 to 12): one month counts as much as one course. "every feature is used, and they are all used the same amount." (CIML §3.2)

<!-- slide 29 -->
## Example: Hours Studied and Day of Birth

<img src="../assets/week9/hours-knn-v0.svg" class="figure figure-plain" style="max-height:440px" alt="Week 8's 30 students by hours studied, failed in a lower row of gray circles and passed in an upper row of orange triangles. A new student who studied 1 hour has 5 nearest neighbors who all failed, so k-NN with k equals 5 predicts fail.">

<img src="../assets/week9/hours-knn-v1.svg" class="figure figure-plain" style="max-height:440px" alt="The same students drawn to scale, with day of month born, 1 to 31, along the bottom and hours studied, 0 to 10, up the side. The new student, born on the 17th, studied 1 hour; their 5 nearest neighbors were born on nearby days: 3 who passed after studying about 7 hours and 2 who failed, so k-NN predicts pass. Dashed lines join the new student to each neighbor.">

Day of birth is noise, but its range (1 to 31) dwarfs hours (0 to 10), so it decides who is "near."

<!-- slide 30 -->
## k-NN: Pros and Cons

**Advantages**

- Does not require training a model (lazy learner)
- Applicable to categorical and numeric attributes.
- no assumptions/criterion about the probability distribution of the data
- can produce decision boundaries of any shape

**Disadvantages**

- can be computationally expensive.
- Easy to overfit (due to noise or high dimensionality)
- Sensitive to redundant or irrelevant attributes
- sensitive to the scale of the data
- No descriptive value, i.e. no meaningful relationship between the class label and the attributes, other than "looks similar to other examples"

<!-- slide 31 -->
# Decision Tree Examples

<!-- slide 32 -->
## Example Tree: Tax Avoidance

<img src="../assets/week9/tax-tree-v0.svg" class="figure figure-plain" style="max-height:420px" alt="A decision tree for tax cheating. Refund: Yes leads to leaf NO; No leads to Marital Status. Marital Status: Single or Divorced leads to Taxable Income; Married leads to leaf NO. Taxable Income: under 80K leads to leaf NO; over 80K leads to leaf YES.">

<img src="../assets/week9/tax-tree-v1.svg" class="figure figure-plain" style="max-height:420px" alt="The same tree with the Refund node filled dark and labeled root node.">

<img src="../assets/week9/tax-tree-v2.svg" class="figure figure-plain" style="max-height:420px" alt="The same tree with Marital Status and Taxable Income filled dark, labeled internal nodes, each splits on a predictor.">

<img src="../assets/week9/tax-tree-v3.svg" class="figure figure-plain" style="max-height:420px" alt="The same tree with the four NO and YES leaves filled dark, labeled leaf nodes, each holds a class label.">

<img src="../assets/week9/tax-tree-v4.svg" class="figure figure-plain" style="max-height:420px" alt="The same tree with every connecting line drawn in orange, labeled branches, connections between nodes.">

<img src="../assets/week9/tax-path-a.svg" class="figure figure-plain" style="max-height:420px" alt="The path Refund, No, Marital Status, Single, Taxable Income, over 80K, YES, drawn in orange. Caption: single, no refund, income 100K, so YES.">

To classify, "follow the decision tree from the root node to a leaf node." ([Bello](https://cs.uic.edu/profiles/gonzalo-bello/), Lecture 12, slides 8–11)

<!-- slide 33 -->
## Example Tree: Hours Studied

<img src="../assets/week9/hours-threshold-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Week 8's 30 students by hours studied: failed in a lower row of gray circles, passed in an upper row of orange triangles.">

<img src="../assets/week9/hours-threshold-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Faint dashed vertical lines at the midpoints between consecutive sorted hours: the candidate thresholds.">

<img src="../assets/week9/hours-threshold-v2.svg" class="figure figure-plain" style="max-height:420px" alt="One solid line at 6.11 hours splits the students: gray tint below, orange tint above, where all 17 passed. A small tree beside it: hours at most 6.11, yes leads to fail, with 3 who passed and 10 who failed; no leads to pass, with 17 who passed and 0 who failed.">

"for real-valued features, the decision tree nodes are parameterized by a feature and a threshold for that feature." (CIML §3.3). 6.11 is scikit-learn's pick.

<!-- slide 34 -->
## Is This a Good Split? (1 of 4)

<img src="../assets/week9/split-quiz-hours-v0.svg" class="figure figure-plain" style="max-height:420px" alt="20 students who passed and 10 who failed, split on Studied more than 6 hours? No: 3 pass, 10 fail. Yes: 17 pass, 0 fail.">

<img src="../assets/week9/split-quiz-hours-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: 77% fail on the no side, 100% pass on the yes side. Good split: each side is mostly one class.">

The split from the last slide.

<!-- slide 35 -->
## Is This a Good Split? (2 of 4)

<img src="../assets/week9/split-quiz-birthday-v0.svg" class="figure figure-plain" style="max-height:420px" alt="20 students who passed and 10 who failed, split on Born on or before the 15th? No: 10 pass, 5 fail. Yes: 10 pass, 5 fail.">

<img src="../assets/week9/split-quiz-birthday-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: 67% pass on both sides. Bad split: each side looks like the whole class, 67% pass.">

Day of birth is random noise.

<!-- slide 36 -->
## Is This a Good Split? (3 of 4)

<img src="../assets/week9/split-quiz-seat-v0.svg" class="figure figure-plain" style="max-height:420px" alt="20 students who passed and 10 who failed, split on Sits on the left side of the room? No: 11 pass, 5 fail. Yes: 9 pass, 5 fail.">

<img src="../assets/week9/split-quiz-seat-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: 69% pass on the no side, 64% pass on the yes side. Bad split: each side looks like the whole class, 67% pass.">

Synthetic data.

<!-- slide 37 -->
## Is This a Good Split? (4 of 4)

<img src="../assets/week9/split-quiz-practice-v0.svg" class="figure figure-plain" style="max-height:420px" alt="20 students who passed and 10 who failed, split on Took the practice exam? No: 2 pass, 9 fail. Yes: 18 pass, 1 fail.">

<img src="../assets/week9/split-quiz-practice-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: 82% fail on the no side, 95% pass on the yes side. Good split: each side is mostly one class.">

Synthetic data.

<!-- slide 38 -->
# Decision Tree Details

<!-- slide 39 -->
## Which Tree?

Sources: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 1](http://ciml.info/dl/v0_99/ciml-v0_99-ch01.pdf), §1.3; Gonzalo Bello Lander, CS 418 Lecture 12 (2026), slide 12; Elena Zheleva, CS 418 lecture slides, Feb. 19, 2026.

- "It is computationally infeasible to consider all of these to try to choose the 'best' one." (CIML)
    - NP-Hard! (according to Prof. Zheleva)
- Instead, "we will build our decision tree greedily. We will begin by asking: If I could only ask one question, what question would I ask?"
-  greedy strategy:
    1. Start with a single node containing all observations in the training set (root node).
    2. Split node into sub-nodes based on an attribute that optimizes certain splitting criterion.
    3. Repeat on each sub-node until reaching certain stopping criterion.

<!-- slide 40 -->
## Scoring a Split: Majority Vote

<img src="../assets/week9/majority-vote-v0.svg" class="figure figure-plain" style="max-height:420px" alt="A bar for 20 Chicago baseball fans: 9 Cubs fans in orange, 11 White Sox fans in gray. Below it, the question: Lives south of Madison Street?">

<img src="../assets/week9/majority-vote-v1.svg" class="figure figure-plain" style="max-height:420px" alt="The bar splits in two. No: 7 Cubs, 3 Sox. Yes: 2 Cubs, 8 Sox.">

<img src="../assets/week9/majority-vote-v2.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: guess Cubs, 7 of 10 correct; guess Sox, 8 of 10 correct. Score equals 7 plus 8 equals 15 out of 20.">

Synthetic data. CIML §1.3: "if I made that guess on the training data, how well would I have done?"

<!-- slide 41 -->
## Majority Vote on the Survey

<img src="../assets/week9/survey-majority-v0.svg" class="figure figure-plain" style="max-height:420px" alt="A bar for the 20 classmates: 6 took CS 412, shown in orange, and 14 did not, in gray. Below it, the question: Took CS 424?">

<img src="../assets/week9/survey-majority-v1.svg" class="figure figure-plain" style="max-height:420px" alt="The bar splits in two. No: 0 took CS 412, 12 did not. Yes: 6 took it, 2 did not.">

<img src="../assets/week9/survey-majority-v2.svg" class="figure figure-plain" style="max-height:420px" alt="Under each child: guess circle, 12 of 12 correct; guess triangle, 6 of 8 correct. Score equals 12 plus 6 equals 18 out of 20.">

"Took CS 424?" (Visualization) is one of the 11 questions we could ask first.

<!-- slide 42 -->
## If You Could Ask Only One Question

<img src="../assets/week9/survey-split-scores-v0.svg" class="figure figure-plain" style="max-height:440px" alt="Every course as a candidate first question, scored by majority vote out of 20. CS 424 and CS 411 tie at 18. CS 488 scores 15. Every other course scores 14, the same as asking no question and guessing circle for everyone. CS 251 is left out because every classmate took it.">

CS 251 splits no one: all 20 took it (it's the CS 418 prerequisite).

<!-- slide 43 -->
## When Majority Vote Can't Decide

<img src="../assets/week9/survey-tie-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Two candidate splits of the same 20 classmates, 6 who took CS 412 and 14 who did not. Took CS 424: no gives 0 and 12, yes gives 6 and 2. Took CS 411: no gives 2 and 14, yes gives 4 and 0. Both get 2 out of 20 wrong.">

<img src="../assets/week9/survey-tie-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added: weighted entropy 0.324 for CS 424, highlighted as lower, and 0.435 for CS 411.">

"Both have the same error rate!!! Which is 'progressing more' towards a lower error?" Elena Zheleva, CS 418 lecture slides, Feb. 19, 2026 (based on Marine Carpuat).

<!-- slide 44 -->
## How Much Does a Split Help?

See also scikit-learn user guide, ["Classification criteria"](https://scikit-learn.org/stable/modules/tree.html#classification-criteria).

- **Majority vote**: guess the majority class on each side and count how many are right; higher is better. (CIML §1.3)
- **Information gain**: the drop in entropy from the split, parent entropy minus the size-weighted entropy of the children; higher is better.
    - $H = -\sum_k p_k \log_2 p_k$. The parent, 20 pass and 10 fail, has $H \approx 0.918$.
- **Gini impurity**: $G = 1 - \sum_k p_k^2$, scikit-learn's default criterion; lower in the children is better.
- We'll talk about these metrics more soon.

<!-- slide 45 -->
## Upload your worksheet

Take a picture of your worksheet, upload it in Canvas

<!-- slide 46 -->
# That's All for Today, See You Wednesday!

<!-- slide 47 -->
# Week 9, Day 2

CS 418 · Week 9 · 🟠 Halsted 🟠

<!-- slide 48 -->
## Map for Week 9, Day 2

- k-NN Review
- Decision Trees for Movie Reviews
- Support Vector Machines
- Kernels
- Quiz

![](../assets/orange-line-map-stops/stop10-halsted.svg)

<!-- slide 49 -->
# k-NN Review

<!-- slide 50 -->
## Similarity as a Fraction

<img src="../assets/week9/similarity-venn.svg" class="figure" style="height:350px; flex-shrink:0" alt="Jaccard and SMC drawn as fractions of Venn diagrams. Each diagram is two circles, a and b, inside a rectangle of all attributes. Jaccard: numerator shades the overlap N11; denominator shades both circles, N10 plus N11 plus N01. SMC: numerator shades the overlap and the space outside both circles, N11 plus N00; denominator shades the whole rectangle.">

- Jaccard: shared 1s out of everything either one has
- SMC: all agreements (shared 1s and shared 0s) out of everything
- Does a shared 0 count?

<!-- slide 51 -->
## k-NN Basic Steps

Sources: Timbers et al., [*Data Science: A First Introduction*, §5.5](https://python.datasciencebook.ca/classification1.html); Bello, Lecture 13. Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218), Week 8's training penguins.

<img src="../assets/week8/cls-penguins-knn-intuition-1-vote.svg" class="figure" style="height:330px; flex-shrink:0" alt="Penguin 1 now has lines to its 5 nearest training penguins inside a small dashed circle, all Gentoo: 5 of 5 Gentoo.">

<img src="../assets/week8/cls-penguins-knn-intuition.svg" class="figure" style="height:330px; flex-shrink:0" alt="Penguin 2 now has lines to its 5 nearest training penguins inside a larger dashed circle that reaches into both clusters: 3 of 5 Gentoo. Penguin 1's 5 of 5 Gentoo vote is still shown below it.">

- Choose a distance and $k$
- find the $k$ nearest training points
- Check the majority vote

<!-- slide 52 -->
## Choosing $k$

Source: Timbers et al., [*Data Science: A First Introduction*, §6.6](https://python.datasciencebook.ca/classification2.html). Data: Week 8's 28 validation penguins.

<img src="../assets/week8/cls-penguins-validation.svg" class="figure" style="height:330px; flex-shrink:0" alt="Three panels of bill length against bill depth, each shading the region k-nearest neighbors predicts as Adélie (gray) or Gentoo (orange), with the 28 validation penguins on top. k = 1: a jagged diagonal boundary, 28 of 28 correct. k = 15: a smoother diagonal boundary, 28 of 28 correct. k = 191: the whole plot is gray, every penguin predicted Adélie, 15 of 28 correct.">

- In general, small $k$ overfits and large $k$ underfits
    - $k = N$ will always...
- Pick $k$ on validation data, never the test set

<!-- slide 53 -->
# Decision Trees and Movie Reviews

<!-- slide 54 -->
## Decision Trees in One Slide

Source: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)*, [ch. 1](http://ciml.info/dl/v0_99/ciml-v0_99-ch01.pdf), §1.3.

- Each node asks about one feature; each leaf is a label
- Tree grows greedily: "If I could only ask one question, what question would I ask?"
- Best question as highest information gain
- `max_depth`: too shallow underfits, too deep overfits

<!-- slide 55 -->
## Demo: Negative, Meh, or Positive?

Data: Hou et al., [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/), `Movies_and_TV`; sample in [`datasets/movie-reviews`](https://github.com/jackbandy/data-science-fun/tree/main/datasets/movie-reviews).

- 4,500 Amazon movie and TV reviews, 1,500 per label
- Labels from stars: 1–2 negative, 3 meh, 4–5 positive
- Each review a vector (of 82 counts)
    - word lists: *bad*, *horrible*, … *ok*, *but*, … *great*, *love*, …
    - negations, length, ALL CAPS, `!`, `?`, `...`

Provenance of the labels: each one comes from the reviewer's star rating, not from someone reading the review. Could some be wrong?

Worksheet checkpoint 3 asks this. Yes, some are wrong: per the dataset README, some 3-star reviews read as glowing, some 1-star reviews are jokes or mistakes, and some reviews are about the disc or the streaming service ("arrived scratched").

<!-- slide 56 -->
## Train the Tree

- Depth 5: 55% train, 51% test
    - guessing?
- Root question: any *not / no / never*?
    - no: mostly positive
    - yes: mostly negative
- Uses only 9 of the 82 clues

DecisionTreeClassifier(max_depth=5, min_samples_leaf=25). Root split: 1,769 training reviews without negation (49% positive), 1,831 with (45% negative). The 9 clues: not/no/never, but, !, great, love, length, waste, ok, okay.

<!-- slide 57 -->
## A Review, From Root to Leaf (1 of 3)

> Absolutely LOVED it!!! Best movie of the year.

- No negation → has `!` → short
- Predicts positive

<img src="../assets/week9/movie-path-loved.svg" class="figure" style="max-height:480px" alt="The review's path down the middle of the tree: has not, no, or never? no. Has an exclamation mark? yes. At least 26 words? no. At least 10 words? no. At least 5 words? yes. Predict positive. At each question, the branch not taken leads off to the side: 11 more leaves predicting negative, meh, or positive; 8 more leaves predicting meh or positive; 3 more leaves predicting positive; 2 more leaves predicting positive; a leaf predicting positive. This leaf is 1 of 26: 67 training reviews, 91% positive, 7% negative, 1% meh.">

<!-- slide 58 -->
## A Review, From Root to Leaf (2 of 3)

> It was ok. Some good parts, but way too long...

- No negation → no `!` → has *but* → short
- Predicts meh

<img src="../assets/week9/movie-path-ok.svg" class="figure" style="max-height:480px" alt="The review's path down the middle of the tree: has not, no, or never? no. Has an exclamation mark? no. Has but? yes. At least 21 words? no. At least 11 words? no. Predict meh. At each question, the branch not taken leads off to the side: 11 more leaves predicting negative, meh, or positive; 7 more leaves predicting positive; 4 more leaves predicting positive; 2 more leaves predicting meh; a leaf predicting meh. This leaf is 1 of 26: 43 training reviews, 86% meh, 12% negative, 2% positive.">

<!-- slide 59 -->
## A Review, From Root to Leaf (3 of 3)

> Horrible. A total waste of two hours.

- No negation → no `!` → no *but* → no *great* → no *love*
- Predicts positive?!

- The tree never asks about *horrible*

<img src="../assets/week9/movie-path-horrible.svg" class="figure" style="max-height:480px" alt="The review's path down the middle of the tree: has not, no, or never? no. Has an exclamation mark? no. Has but? no. Has great? no. Has love? no. Predict positive. At each question, the branch not taken leads off to the side: 11 more leaves predicting negative, meh, or positive; 7 more leaves predicting positive; 4 more leaves predicting meh; 2 more leaves predicting positive; a leaf predicting positive. This leaf is 1 of 26: 913 training reviews, 38% positive, 34% meh, 28% negative.">

Few reviews use "horrible", so the greedy splits pick broader clues; "waste" is only asked on the negation side. Same lesson as Monday's CS 424 vs. CS 411 split.

<!-- slide 60 -->
## Try the Demo in the Python Yard

Worksheet checkpoint:

- Open [`movie-reviews.ipynb`](https://dodatascience.fun/yard/notebooks/index.html?path=movie-reviews.ipynb)
- Add three words to a list and rerun
    - new root question?
    - new test accuracy?
- You can also try testing on your own review

<iframe src="https://dodatascience.fun/timer/" title="CTA-style countdown timer" loading="lazy" style="width: 100%; height: 420px; display: block; border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

<!-- slide 61 -->
## Same Clues, Other Classifiers

Test accuracy on the same 82 clues:

- decision tree: 51%
- $k$-NN ($k = 15$, standardized): 60%
- linear SVM: 64%

What is an SVM?

<!-- slide 62 -->
# Support Vector Machines

<!-- slide 63 -->
## Which Line Would You Pick?

Based on Bello, Lecture 13, Exercise 13.3, and [VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html). Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218), 28 of Week 8's penguins.

<img src="../assets/week9/svm-lines-v0.svg" class="figure" style="max-height:420px" alt="28 penguins, bill length against bill depth: Adélie as gray circles at upper left, Gentoo as orange triangles at lower right, with an empty gap between them.">

<img src="../assets/week9/svm-lines-v1.svg" class="figure" style="max-height:420px" alt="Three lines, A, B, and C, each with every Gentoo on one side and every Adélie on the other. A runs close to the Gentoo; C is tilted; B runs through the middle of the gap.">

Homework 10 (`svm-pick-max-margin-line`) is this question.

<!-- slide 64 -->
## SVM Goal: Maximize the Margin

Sources: [VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html); Daumé, CIML [ch. 7](http://ciml.info/dl/v0_99/ciml-v0_99-ch07.pdf), §7.7. Axes are standardized and drawn at equal scale.

<img src="../assets/week9/svm-lines-v2.svg" class="figure" style="max-height:400px" alt="Line B in orange with a gray band around it reaching the closest penguin on each side, edged by dashed lines. Distances to the closest penguin, in standard units: A 0.19, B 0.47, C 0.20.">

- Margin: the empty band around the line; the SVM picks the widest

<!-- slide 65 -->
## Support Vectors

Source: [VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html).

<img src="../assets/week9/svm-lines-v3.svg" class="figure" style="max-height:400px" alt="The same margin, with the three penguins that touch its dashed edges circled: one Adélie and two Gentoo.">

- "Support vectors" are points touching the margin
- they decide where the line goes
- k-NN keeps every training point; an SVM keeps only these

<!-- slide 66 -->
## What if a does not work?

<img src="../assets/week9/kernel-lift-v0.svg" class="figure" style="max-height:420px" alt="Points on a number line from minus 4 to 4: circles at both ends, triangles in the middle. Dashed lines at minus 2.5 and 2.5 show that two cuts are needed.">

<img src="../assets/week9/kernel-lift-v1.svg" class="figure" style="max-height:420px" alt="The same points lifted onto the parabola y equals x squared. The triangles sit low, the circles high, and an orange horizontal line at x squared equals 6.5 separates them.">

- One feature: no single cut separates the classes

<!-- slide 67 -->
## Kernels

Based on [VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html); see also Daumé, CIML [ch. 11](http://ciml.info/dl/v0_99/ciml-v0_99-ch11.pdf). Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218), all 342 measured penguins.

<img src="../assets/week9/kernel-penguins-v0.svg" class="figure" style="max-height:400px" alt="All 342 measured penguins: Chinstrap as orange triangles at upper right, between the Adélie at upper left and the Gentoo at lower right, both gray circles.">

<img src="../assets/week9/kernel-penguins-v1.svg" class="figure" style="max-height:400px" alt="A linear SVM's boundary: one straight diagonal line, with Gentoo on the Chinstrap side; 88 percent training accuracy.">

<img src="../assets/week9/kernel-penguins-v2.svg" class="figure" style="max-height:400px" alt="An RBF kernel SVM's boundary: a closed curve around the Chinstraps; 96 percent training accuracy.">

- Chinstraps sit between the other two species
- no straight line to fence them off
- Kernel trick: the benefit of new features without computing them; `SVC(kernel="rbf")`

<!-- slide 68 -->
## Common Kernels

Sources: Bello, Lecture 13, slide 27 ("Kernel Trick"); scikit-learn user guide, [§1.4, "Kernel functions"](https://scikit-learn.org/stable/modules/svm.html). Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218), Chinstrap vs. the rest; training accuracy above each panel.

<img data-id="kernel-types" src="../assets/week9/kernel-types.svg" class="figure" alt="Four panels of the Chinstrap-versus-rest penguins, each shaded by a different SVM kernel. Linear: one straight line, 88 percent. Polynomial, degree 3: a curved boundary around the Chinstraps, 97 percent. RBF: a closed pocket around the Chinstraps, 96 percent. Sigmoid: a tangled boundary that cuts through both groups, 75 percent.">

<!-- slide 69 -->
## Common Kernels

Sources: Bello, Lecture 13, slide 27 ("Kernel Trick"); scikit-learn user guide, [§1.4, "Kernel functions"](https://scikit-learn.org/stable/modules/svm.html). Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218), Chinstrap vs. the rest; training accuracy above each panel.

<img data-id="kernel-types" src="../assets/week9/kernel-types.svg" class="figure" style="flex-shrink:0" alt="Four panels of the Chinstrap-versus-rest penguins, each shaded by a different SVM kernel. Linear: one straight line, 88 percent. Polynomial, degree 3: a curved boundary around the Chinstraps, 97 percent. RBF: a closed pocket around the Chinstraps, 96 percent. Sigmoid: a tangled boundary that cuts through both groups, 75 percent.">

<!-- Raw HTML so each row can be a fragment; processing is off so Quarto keeps the row classes. -->
<table data-quarto-disable-processing="true">
<thead><tr><th>Kernel</th><th><i>K</i>(<b>u</b>, <b>v</b>)</th><th>Boundary</th><th>scikit-learn</th></tr></thead>
<tbody>
<tr class="fragment"><td>linear</td><td><b>u</b><sup><i>T</i></sup><b>v</b></td><td>a straight line</td><td><code>SVC(kernel=&quot;linear&quot;)</code></td></tr>
<tr class="fragment"><td>polynomial</td><td>(<i>γ</i> <b>u</b><sup><i>T</i></sup><b>v</b> + <i>r</i>)<sup><i>d</i></sup></td><td>curves of degree <i>d</i></td><td><code>SVC(kernel=&quot;poly&quot;, degree=3)</code></td></tr>
<tr class="fragment"><td>RBF</td><td>exp(−<i>γ</i> ‖<b>u</b> − <b>v</b>‖<sup>2</sup>)</td><td>any shape; similarity drops with distance</td><td><code>SVC(kernel=&quot;rbf&quot;)</code>, the default</td></tr>
<tr class="fragment"><td>sigmoid</td><td>tanh(<i>γ</i> <b>u</b><sup><i>T</i></sup><b>v</b> + <i>r</i>)</td><td>rarely the best choice</td><td><code>SVC(kernel=&quot;sigmoid&quot;)</code></td></tr>
</tbody>
</table>

The polynomial panel is degree 3 with coef0 = 1 (the r in the formula). RBF is scikit-learn's default and the usual first choice.

<!-- slide 70 -->
## Linear or Kernel? (1 of 5)

<img src="../assets/week9/kernel-shape-a-v0.svg" class="figure" style="max-height:440px" alt="Shape A: two separate clouds. Linear or kernel?">

<img src="../assets/week9/kernel-shape-a-v1.svg" class="figure" style="max-height:440px" alt="Shape A, answer: linear. A straight line separates the two clouds.">

Show of hands before each reveal. Homework 10 (`kernel-needed`) uses these five shape kinds.

<!-- slide 71 -->
## Linear or Kernel? (2 of 5)

<img src="../assets/week9/kernel-shape-b-v0.svg" class="figure" style="max-height:440px" alt="Shape B: a ring of circles around a ring of triangles. Linear or kernel?">

<img src="../assets/week9/kernel-shape-b-v1.svg" class="figure" style="max-height:440px" alt="Shape B, answer: kernel. A closed curve around the inner ring separates them.">

<!-- slide 72 -->
## Linear or Kernel? (3 of 5)

<img src="../assets/week9/kernel-shape-c-v0.svg" class="figure" style="max-height:440px" alt="Shape C: two long parallel bands. Linear or kernel?">

<img src="../assets/week9/kernel-shape-c-v1.svg" class="figure" style="max-height:440px" alt="Shape C, answer: linear. A straight line runs between the bands.">

<!-- slide 73 -->
## Linear or Kernel? (4 of 5)

<img src="../assets/week9/kernel-shape-d-v0.svg" class="figure" style="max-height:440px" alt="Shape D: XOR corners: triangles in two opposite corners, circles in the other two. Linear or kernel?">

<img src="../assets/week9/kernel-shape-d-v1.svg" class="figure" style="max-height:440px" alt="Shape D, answer: kernel. Curved boundaries split the four corners.">

<!-- slide 74 -->
## Linear or Kernel? (5 of 5)

<img src="../assets/week9/kernel-shape-e-v0.svg" class="figure" style="max-height:440px" alt="Shape E: two interlocking moons. Linear or kernel?">

<img src="../assets/week9/kernel-shape-e-v1.svg" class="figure" style="max-height:440px" alt="Shape E, answer: kernel. A curved boundary follows the gap between the moons.">

<!-- slide 75 -->
## $\gamma$: How Far Each Point Reaches

Source: scikit-learn, ["RBF SVM parameters"](https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html). Data: [Palmer Penguins](https://doi.org/10.5281/zenodo.3960218); half the penguins train, half are held out.

<img src="../assets/week9/svm-gamma-v0.svg" class="figure" style="height:290px; flex-shrink:0" alt="Chinstrap against the rest with gamma equals 0.05: a broad curve that misses some Chinstraps. Train 89 percent, test 90 percent.">

<img src="../assets/week9/svm-gamma-v1.svg" class="figure" style="height:290px; flex-shrink:0" alt="Gamma equals 1: a pocket around the Chinstraps. Train 98 percent, test 94 percent.">

<img src="../assets/week9/svm-gamma-v2.svg" class="figure" style="height:290px; flex-shrink:0" alt="Gamma equals 50: small islands around single penguins. Train 100 percent, test 85 percent.">

- $\gamma$ sets how fast RBF similarity, $\exp(-\gamma \cdot \text{distance}^2)$, fades
- Small $\gamma$: each penguin's influence reaches far → smooth boundary (underfit)
- Large $\gamma$: influence stays local → islands around single penguins (overfit)
- Tune $C$ and $\gamma$ on validation data, like $k$ and `max_depth`

<!-- slide 76 -->
## SVMs on the Movie Reviews

```python
model = make_pipeline(StandardScaler(), LinearSVC())
```

- 64% test (tree: 51%)
- Biggest weights: *not/no/never*, *horrible* (negative); *great*, *love* (positive)
- Unlike the tree, it uses *horrible*

Scale first, as with k-NN: scikit-learn's user guide (§1.4) says SVMs "are not scale invariant." Negative vs. positive only: tree 76%, linear SVM 81%.

<!-- slide 77 -->
## SVMs: Pros and Cons

Sources: [VanderPlas](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html); Bello, Lecture 13.

- Pros: compact (support vectors only), fast predictions, kernels handle curved boundaries
- Cons: slow to train on big data, sensitive to $C$ and $\gamma$, no probabilities

<!-- slide 78 -->
# Wrap-Up

<!-- slide 79 -->
## Four Classifiers in Brief

<!-- Raw HTML so each row can be a fragment; processing is off so Quarto keeps the row classes. -->
<table data-quarto-disable-processing="true">
<thead><tr><th>Classifier</th><th>Boundary</th><th>Hyperparameters</th><th>Strength</th><th>Weakness</th></tr></thead>
<tbody>
<tr class="fragment"><td><i>k</i>-NN</td><td>any shape</td><td><i>k</i></td><td>no training; flexible</td><td>slow to predict; scale-sensitive</td></tr>
<tr class="fragment"><td>Decision tree</td><td>rectilinear</td><td>max depth</td><td>interpretable</td><td>easy to overfit</td></tr>
<tr class="fragment"><td>SVM</td><td>linear, or curved with a kernel</td><td><i>C</i>, kernel, <i>γ</i></td><td>maximum margin</td><td>slow to train</td></tr>
<tr class="fragment"><td>Naive Bayes</td><td>follows the class distributions</td><td>(few)</td><td>probabilities</td><td>assumes independence</td></tr>
</tbody>
</table>

Same `fit` / `predict` / `score` for all four.

From Week 8. Keep the Naive Bayes column: Homework 10's `classifier-identify-description` describes Naive Bayes one time in four.

<!-- slide 80 -->
## Upload your worksheet

Take a picture of your worksheet, upload it in Canvas

<!-- slide 81 -->
## Quiz Time

Week 9 Quiz: the last 30 minutes of class

<!-- slide 82 -->
# That's All for Today, See You Monday!

<!-- slide 83 -->
# Appendix: Entropy and Information Gain

<!-- slide 84 -->
## Entropy: How Much Does a Split Help Distinguish Classes?

Source: Gonzalo Bello Lander, CS 418 Lecture 12, "Classification (I)" (2026), slides 14–16. Figure based on Sebastian Raschka, "[Why are we growing decision trees via entropy instead of the classification error?](https://sebastianraschka.com/faq/docs/decisiontree-error-vs-entropy.html)": "if we have 2 entropy values (left and right child node), the average will fall onto the straight, connecting line."

"Entropy measures the impurity of a node of the tree."

$$H(v_m) = -\sum_k p(y_k \mid v_m) \log_2 p(y_k \mid v_m)$$

"where $p(y_k \mid v_m)$ is the proportion of observations from class $y_k$ at node $v_m$."

- (or [`scipy.stats.entropy`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.entropy.html), with `base=2` 🙂)

- "Minimum value: 0. When all observations belong to the same class."
- "Maximum value: $\log_2 k$ where $k$ is the number of classes. When observations are equally distributed among all classes."

<img src="../assets/week9/survey-entropy-curve-v0.svg" class="figure figure-plain" style="max-height:400px" alt="Binary entropy curve: zero at share 0 and share 1, rising to a peak of 1 at share 0.5.">

<img src="../assets/week9/survey-entropy-curve-v1.svg" class="figure figure-plain" style="max-height:400px" alt="The parent node, 6 who took CS 412 and 14 who did not, marked on the curve at share 0.3, entropy 0.881.">

<img src="../assets/week9/survey-entropy-curve-v2.svg" class="figure figure-plain" style="max-height:400px" alt="The CS 411 children in gray at shares 0.125 and 1, joined by a dashed line. Their weighted average, 0.435, sits on that line directly below the parent; the drop, 0.446, is bracketed.">

<img src="../assets/week9/survey-entropy-curve-v3.svg" class="figure figure-plain" style="max-height:400px" alt="Added in orange: the CS 424 children at shares 0 and 0.75, joined by a dashed line. Their weighted average, 0.324, sits lower, so the drop from the parent, 0.557, is larger than CS 411's.">

<!-- slide 85 -->
## Information Gain

"Information gain measures the difference in entropy resulting from splitting a node of the tree on a given attribute."

$$IG(X_i, v_m) = H(v_m) - \sum_{h=1}^{M} \frac{N_h}{N} H(v_h)$$

"where $N_h$ is the number of observations at node $v_h$."

- (or [`sklearn.metrics.mutual_info_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mutual_info_score.html), which reports it in nats, not bits 🙂)

- Parent entropy minus the size-weighted average entropy of the children.
- "The best attribute to split on is the one that maximizes the information gain."
- "Information gain is the splitting criterion function used by the ID3 and the C4.5 algorithms."

<!-- slide 86 -->
## How Much Does This Question Help Us Distinguish Between Classes?

<img src="../assets/week9/split-entropy-v0.svg" class="figure figure-plain" style="max-height:420px" alt="The Cubs and Sox split from the majority-vote slide: 9 Cubs and 11 Sox, split on Lives south of Madison Street? No: 7 Cubs, 3 Sox. Yes: 2 Cubs, 8 Sox.">

<img src="../assets/week9/split-entropy-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added beside the parent: entropy 0.993, close to the maximum of 1 because the parent is nearly 50/50.">

<img src="../assets/week9/split-entropy-v2.svg" class="figure figure-plain" style="max-height:420px" alt="Added under each child: entropy 0.881 for 7 Cubs and 3 Sox, entropy 0.722 for 2 Cubs and 8 Sox.">

<img src="../assets/week9/split-entropy-v3.svg" class="figure figure-plain" style="max-height:420px" alt="Added in orange: information gain equals 0.993 minus one half of 0.881 minus one half of 0.722, which is 0.191.">

<!-- slide 87 -->
## How Much Does This Question Help Us Distinguish Between Classes?

**Parent**, 9 Cubs / 11 Sox:

$$H = -\tfrac{9}{20}\log_2\tfrac{9}{20} - \tfrac{11}{20}\log_2\tfrac{11}{20} \approx 0.993$$

- **No** (north of Madison), 7 Cubs / 3 Sox: $H = -\tfrac{7}{10}\log_2\tfrac{7}{10} - \tfrac{3}{10}\log_2\tfrac{3}{10} \approx 0.881$
- **Yes** (south of Madison), 2 Cubs / 8 Sox: $H = -\tfrac{2}{10}\log_2\tfrac{2}{10} - \tfrac{8}{10}\log_2\tfrac{8}{10} \approx 0.722$
- **Weighted average** of the children, 10 of 20 fans each: $\tfrac{10}{20}(0.881) + \tfrac{10}{20}(0.722) \approx 0.802$
- **Information gain**: $0.993 - 0.802 \approx 0.191$

<!-- slide 88 -->
## Information Gain for "Took CS 424?"

<img src="../assets/week9/survey-split-entropy-v0.svg" class="figure figure-plain" style="max-height:420px" alt="The survey split: 6 who took CS 412 and 14 who did not, split on Took CS 424? No: 0 and 12. Yes: 6 and 2.">

<img src="../assets/week9/survey-split-entropy-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added beside the parent: entropy 0.881.">

<img src="../assets/week9/survey-split-entropy-v2.svg" class="figure figure-plain" style="max-height:420px" alt="Added under each child: entropy 0.000 for the pure no side, 0.811 for the yes side.">

<img src="../assets/week9/survey-split-entropy-v3.svg" class="figure figure-plain" style="max-height:420px" alt="Added in orange: information gain equals 0.881 minus 12 twentieths times 0.000 minus 8 twentieths times 0.811, which is 0.557.">

Unequal children (12 and 8), so the weights are $\tfrac{12}{20}$ and $\tfrac{8}{20}$.

<!-- slide 89 -->
## The Greedy Choice

<img src="../assets/week9/survey-split-scores-v0.svg" class="figure figure-plain" style="max-height:440px" alt="Every course scored by majority vote: CS 424 and CS 411 tie at 18 out of 20.">

<img src="../assets/week9/survey-split-scores-v1.svg" class="figure figure-plain" style="max-height:440px" alt="Added on the right: information gain for every course. CS 424 is highest at 0.557, then CS 411 at 0.446, CS 301 at 0.108, and the rest below 0.1. CS 361 gains nothing.">

The root asks "Took CS 424?"; the same search then repeats inside each child.

<!-- slide 90 -->
## Worked Example

<img src="../assets/week9/ex12-4-v0.svg" class="figure figure-plain" style="max-height:400px" alt="The two candidate splits with counts only.">

<img src="../assets/week9/ex12-4-v1.svg" class="figure figure-plain" style="max-height:400px" alt="Child entropies added: 0.971 for both X a children, 0.722 for both X b children.">

<img src="../assets/week9/ex12-4-v2.svg" class="figure figure-plain" style="max-height:400px" alt="Information gain added: X a gives 1 minus one half of 0.971 minus one half of 0.971, equals 0.029. X b, in orange, gives 1 minus one half of 0.722 minus one half of 0.722, equals 0.278.">

Source: Gonzalo Bello Lander, CS 418 Lecture 12, "Classification (I)" (2026), slides 16–18. Parent: $H(v_0) = -\frac{5}{10}\log_2\frac{5}{10} - \frac{5}{10}\log_2\frac{5}{10} = 1$. "Splitting on $X_b$ is better than splitting on $X_a$."

<!-- slide 91 -->
# Appendix: Growing and Tuning a Tree

<!-- slide 92 -->
## Growing the Tree, One Level at a Time

<img src="../assets/week9/survey-tree-grow-v0.svg" class="figure figure-plain" style="max-height:430px" alt="Depth 0: a single leaf holding all 20 classmates, 6 who took CS 412 and 14 who did not. It predicts circle for everyone: 70 percent training accuracy.">

<img src="../assets/week9/survey-tree-grow-v1.svg" class="figure figure-plain" style="max-height:430px" alt="Depth 1: the root asks Took CS 424? No leads to a leaf with 0 and 12, predicting circle. Yes leads to a leaf with 6 and 2, predicting triangle. 90 percent training accuracy.">

<img src="../assets/week9/survey-tree-grow-v2.svg" class="figure figure-plain" style="max-height:430px" alt="Depth 2: the yes side asks Took CS 411? No gives a 2 and 2 tie, predicting circle; yes gives 4 and 0, predicting triangle. Still 90 percent training accuracy.">

<img src="../assets/week9/survey-tree-grow-v3.svg" class="figure figure-plain" style="max-height:430px" alt="The full tree, depth 4: under the 2 and 2 tie it asks Took CS 427? and then Took CS 488?, ending in leaves of 1 and 0, 0 and 2, and 1 and 0. 100 percent training accuracy.">

To separate the last four classmates, the full tree asks about CS 427 and CS 488 (tied at that node; scikit-learn picked CS 427).

<!-- slide 93 -->
## Classify Alice: Walk From Root to Leaf

<img src="../assets/week9/survey-tree-path.svg" class="figure figure-plain" style="max-height:320px" alt="The full survey tree with Alice's path in orange: Took CS 424? No, to the leaf with 0 who took CS 412 and 12 who did not. The tree predicts circle after one question.">

- One question, and Alice lands in a leaf where none of the 12 classmates took CS 412. Correct!
- Had the root been "Took CS 411?", Alice (who took CS 411) would land in the 4 ▲ / 0 ● leaf, and the tree would predict ▲.

<!-- slide 94 -->
## Empty Tree vs. Full Tree

Source: Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)* (CIML), v0.99, [ch. 2](http://ciml.info/dl/v0_99/ciml-v0_99-ch02.pdf)

- **Empty tree**: make the same prediction regardless of input
    - not allowed to ask any questions about input
- **Full tree**: For the leaves corresponding to training points, the full decision tree will always make the correct prediction
- "Of course our goal is not to build a model that gets 0% error on the training data. This would be easy! Our goal is a model that will do well on future, unseen data."

<!-- slide 95 -->
## Max Depth Is a Hyperparameter

<img src="../assets/week9/depth-accuracy-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Training accuracy against max depth from 0 to 15: it climbs from about 53 percent to 100 percent and never goes down.">

<img src="../assets/week9/depth-accuracy-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added in orange, held-out accuracy: it rises to about 79 percent near depth 3, then slowly falls to about 73 percent.">

<img src="../assets/week9/depth-accuracy-v2.svg" class="figure figure-plain" style="max-height:420px" alt="The peak of the held-out curve is circled, labeled best held-out: depth 3. Left end labeled underfit, empty tree; right end labeled overfit, near-full tree.">

"By playing with this depth knob, we can adjust between underfitting (the empty tree, depth = 0) and overfitting (the full tree, depth = ∞). Such a knob is called a hyperparameter." (CIML §2.6). Synthetic data, 10% labels flipped; average of 40 draws of 150 training and 1,500 held-out points.

<!-- slide 96 -->
## Choosing Depth on a Bigger Class

<img src="../assets/week9/survey-depth-accuracy-v0.svg" class="figure figure-plain" style="max-height:420px" alt="Training accuracy against max depth from 0 to 10: it climbs from about 71 percent to about 88 percent and never goes down.">

<img src="../assets/week9/survey-depth-accuracy-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Added in orange, validation accuracy: it rises to about 87 percent at depth 3, then falls to about 80 percent.">

<img src="../assets/week9/survey-depth-accuracy-v2.svg" class="figure figure-plain" style="max-height:420px" alt="The peak of the validation curve is circled, labeled best validation: depth 3. Left end labeled underfit, empty tree; right end labeled overfit, deep tree.">

The code demo's 1,000 synthetic students: 600 training, 200 validation, entropy splits.

<!-- slide 97 -->
## Tuning Max Depth Without Cheating

- Tune on training data?
- Tune on test data?
    - "it breaks our cardinal rule about test data: that you should never touch your test data." (CIML)
- Instead: "Split your data into 70% training data, 10% development data and 20% test data." Pick the depth that does best on the development (validation) data. (CIML)
- "Never ever touch your test data during learning!" (Zheleva)

<!-- slide 98 -->
## Occam's Razor

> Among competing hypotheses, the one that makes the fewest assumptions should be selected.

A common paraphrase of William of Ockham (c. 1287–1347)

<!-- slide 99 -->
## R2D3: A Visual Introduction to Machine Learning

<iframe src="https://r2d3.us/visual-intro-to-machine-learning-part-1/" title="R2D3, A Visual Introduction to Machine Learning, Part 1" loading="lazy" style="border: 1px solid #999; box-shadow: 0 10px 28px rgba(0,0,0,0.12); border-radius: 3px;"></iframe>

Stephanie Yee and Tony Chu, "A Visual Introduction to Machine Learning, Part 1," R2D3: <https://r2d3.us/visual-intro-to-machine-learning-part-1/>

<!-- slide 100 -->
## Decision Trees: Pros and Cons

Jake VanderPlas, *[Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/05.08-random-forests.html)*, "In-Depth: Decision Trees and Random Forests."

**Advantages**

- Decision trees are very easy to understand and interpret.
- Algorithms for decision tree induction are very efficient, esp. greedy algorithms
- Usually robust to irrelevant / redundant attributes.

**Disadvantages** (Bello)

- Easy to overfit.
    - Small changes in the data can lead to large changes in the structure of decision trees.
- Rectilinear decision boundaries.

<img src="../assets/week9/knn-vs-tree.svg" class="figure figure-plain" style="max-height:300px" alt="The 60-point data set twice. Left, k-NN with k equals 7: a curving, wiggly boundary. Right, a depth-3 decision tree: a boundary made of horizontal and vertical segments.">

"it is very easy to go too deep in the tree, and thus to fit details of the particular data rather than the overall properties of the distributions they are drawn from." (VanderPlas)

<!-- slide 101 -->
# Appendix: Code Demo

<!-- slide 102 -->
## The Plan

- Same survey, bigger class: 1,000 synthetic students.
- Split them: 60% training, 20% validation, 20% test.
- $k$-NN: pick $k$ on the validation set.
- Decision tree: pick `max_depth` on the validation set.
- Touch the test set once, at the very end.
- Run it yourself: [Python Yard (beta)](https://dodatascience.fun/yard/notebooks/index.html?path=week9.ipynb), notebook `week9.ipynb`.

<!-- slide 103 -->
## Simulate the Survey

```python
def make_survey(n, seed=418):
    rng = np.random.default_rng(seed)
    senior = rng.random(n) < 0.5   # hidden: further along in the major
    ai_fan = rng.random(n) < 0.45  # hidden: into AI and data
    ...
    survey = {
        "CS 411": took(np.where(ai_fan, 0.85, 0.05) * np.where(senior, 1, 0.7)),
        "CS 424": took(np.where(ai_fan, 0.6, 0.1)),
        ...                                     # 9 more courses
    }
    survey["CS 412"] = took(np.where(ai_fan, 0.9, 0.03) * np.where(senior, 1, 0.5))
```

In [ ]:
import numpy as np
import polars as pl

def make_survey(n, seed=418):
    rng = np.random.default_rng(seed)
    senior = rng.random(n) < 0.5   # hidden: further along in the major
    ai_fan = rng.random(n) < 0.45  # hidden: into AI and data
    gamer = rng.random(n) < 0.25   # hidden: into games and art
    def took(p):                   # 1 with probability p, else 0
        return (rng.random(n) < p).astype(int)
    survey = {
        "CS 100": took(0.08),
        "CS 251": np.ones(n, int),  # the prerequisite for CS 418
        "CS 301": took(np.where(senior, 0.85, 0.35)),
        "CS 361": took(np.where(senior, 0.8, 0.3)),
        "CS 401": took(np.where(senior, 0.75, 0.15)),
        "CS 411": took(np.where(ai_fan, 0.85, 0.05) * np.where(senior, 1, 0.7)),
        "CS 424": took(np.where(ai_fan, 0.6, 0.1)),
        "CS 426": took(np.where(gamer, 0.6, 0.05)),
        "CS 427": took(np.where(gamer, 0.5, 0.05)),
        "CS 480": took(np.where(senior, 0.5, 0.2)),
        "CS 488": took(0.12),
    }
    survey["CS 412"] = took(np.where(ai_fan, 0.9, 0.03) * np.where(senior, 1, 0.5))
    return pl.DataFrame(survey)

- Each student gets hidden traits; each course is a coin flip whose odds depend on them.
- CS 412 depends on the hidden traits, never directly on other courses. The models only see the courses.
- Full code: `make_survey` in [`week9.ipynb`](https://dodatascience.fun/yard/notebooks/index.html?path=week9.ipynb).

<!-- slide 104 -->
## The Survey as a Table

In [ ]:
survey = make_survey(1000)
survey.head(5)

<!-- slide 105 -->
## Features, Label, and Three Sets

In [ ]:
from sklearn.model_selection import train_test_split

courses = [c for c in survey.columns if c != "CS 412"]
X = survey.select(courses).to_numpy().astype(bool)  # took it or not, 11 courses
y = survey["CS 412"].to_numpy()                      # the label: took CS 412?

# 20% test, then 25% of the rest (20% overall) for validation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=0)
print(len(y_train), "training,", len(y_val), "validation,", len(y_test), "test")

<!-- slide 106 -->
## k-NN in Three Lines

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric="jaccard")
knn.fit(X_train, y_train)                 # lazy: this just stores the training data
print(f"validation accuracy: {knn.score(X_val, y_val):.1%}")

- `metric="jaccard"`: shared 0s don't count toward similarity.
- The default metric is Minkowski with $p = 2$, which is Euclidean distance.

<!-- slide 107 -->
## Choose k on the Validation Set

In [ ]:
import matplotlib.pyplot as plt

ks = [1, 3, 5, 9, 15, 25, 51, 101, 201, 401]
val_acc = []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k, metric="jaccard").fit(X_train, y_train)
    val_acc.append(model.score(X_val, y_val))
best_k = ks[int(np.argmax(val_acc))]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(ks, val_acc, marker="o", color="#f9461c")
ax.set(xscale="log", xlabel="k", ylabel="validation accuracy", title=f"best k = {best_k}")
plt.show()

<!-- slide 108 -->
## A Decision Tree in Three Lines

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(criterion="entropy", max_depth=2, random_state=0)
tree.fit(X_train, y_train)                # greedy: best question first
fig, ax = plt.subplots(figsize=(11, 3.6))
plot_tree(tree, feature_names=courses, class_names=["didn't take 412", "took 412"], ax=ax)
plt.show()

With 600 students the root asks about CS 411, not CS 424 as in our class of 20. "Small changes in the data can lead to large changes in the structure of decision trees." (Bello)

<!-- slide 109 -->
## Choose max_depth on the Validation Set

In [ ]:
depths = range(1, 11)
trees = [DecisionTreeClassifier(criterion="entropy", max_depth=d, random_state=0)
         .fit(X_train, y_train) for d in depths]
train_acc = [t.score(X_train, y_train) for t in trees]
val_acc = [t.score(X_val, y_val) for t in trees]
best_depth = depths[int(np.argmax(val_acc))]

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(depths, train_acc, marker="o", color="#565a5c", label="training")
ax.plot(depths, val_acc, marker="o", color="#f9461c", label="validation")
ax.set(xlabel="max_depth", ylabel="accuracy", title=f"best depth = {best_depth}")
ax.legend()
plt.show()

<!-- slide 110 -->
## Test Once, Then Ask About Alice

In [ ]:
final_knn = KNeighborsClassifier(n_neighbors=best_k, metric="jaccard")
final_tree = DecisionTreeClassifier(
    criterion="entropy", max_depth=best_depth, random_state=0)
for model in (final_knn, final_tree):
    model.fit(X_train, y_train)
    print(f"{type(model).__name__}: test accuracy {model.score(X_test, y_test):.1%}")

# Alice took CS 251, 401, 411, and 480
alice = np.array([[c in ("CS 251", "CS 401", "CS 411", "CS 480") for c in courses]])
print("k-NN:", final_knn.predict(alice)[0], "  tree:", final_tree.predict(alice)[0])

- `1` means "took CS 412", `0` means "didn't".
- Try it with your own row from the worksheet!

<!-- slide 111 -->
## Decision Trees in scikit-learn

```python
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(criterion="entropy", max_depth=3)
model.fit(X_train, y_train)        # learn the splits
model.predict(X_test)              # walk each row from root to leaf
model.score(X_test, y_test)        # accuracy on held-out data
```

- `criterion`: default is `"gini"`
    - `"entropy"` and `"log_loss"`
- [`max_depth`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html): default `None`. "If None, then nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples." The default is the full tree.

<!-- slide 112 -->
# Appendix: Classification Review

<!-- slide 113 -->
## Training, Testing, Validation, and Development Sets

TODO

- **TK**: training / testing / validation / development sets

<!-- slide 114 -->
## Review: The Classification Setup

**predictors $X$** &nbsp;⟶&nbsp; classifier &nbsp;⟶&nbsp; **class label $y$**

- A classification model (or classifier) takes as input the values of the predictors and returns as output the class label.
- **Binary**: "trying to predict a simple yes/no response. For instance, predict whether Alice will enjoy a course or not." (CIML)
- **Multiclass**: "trying to put an example into one of a number of classes. For instance, predict whether a news story is about entertainment, sports, politics, religion, etc." (CIML)

<!-- slide 115 -->
## Review: The Confusion Matrix

Source: Gonzalo Bello Lander, CS 418 Lecture 12, "Classification (I)" (2026), slides 3–5.

|  | Predicted P ($y=1$) | Predicted N ($y=0$) |
|---|:---:|:---:|
| **True P ($y=1$)** | TP | FN |
| **True N ($y=0$)** | FP | TN |

- Accuracy $= \frac{TP+TN}{TP+FP+TN+FN}$: "how many observations were classified correctly"
    - (or [`sklearn.metrics.accuracy_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) 🙂)
- Precision $= \frac{TP}{TP+FP}$: "how many observations classified as positive were indeed positive"
    - (or [`sklearn.metrics.precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html) 🙂)
- Recall $= \frac{TP}{TP+FN}$: "how many positive observations were classified correctly"
    - (or [`sklearn.metrics.recall_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html) 🙂)
- $F_1 = \frac{2 \cdot \text{Recall} \cdot \text{Precision}}{\text{Recall} + \text{Precision}}$: "measures trade-off between precision and recall"
    - (or [`sklearn.metrics.f1_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html) 🙂)

<!-- slide 116 -->
# Appendix: More Examples

<!-- slide 117 -->
## Example: k-NN in Two Dimensions

<img src="../assets/week9/knn-steps-v3.svg" class="figure figure-plain" style="max-height:440px" alt="Seventeen training points: orange triangles upper right, gray circles lower left, one stray circle inside the triangle cluster. A small circle around the question mark holds only its single nearest neighbor, the stray circle, so the question mark becomes a gray circle.">

<img src="../assets/week9/knn-steps-v4.svg" class="figure figure-plain" style="max-height:440px" alt="A larger circle holds the three nearest neighbors, the stray circle and two triangles, so the question mark becomes an orange triangle.">

Synthetic, based on CIML Fig. 3.4, where the test point "should be positive, but its NN is actually a negative point that's noisy."

<!-- slide 118 -->
## Example: Lazy Learning, the Data Is the Model

<img src="../assets/week9/knn-lazy-v3.svg" class="figure figure-plain" style="max-height:440px" alt="Seventeen training points with the 1-nearest-neighbor decision boundary: jagged, with a notch around the stray circle inside the triangle cluster.">

"This line is called the decision boundary for this classifier. It is the line with positive land on one side and negative land on the other side." (CIML §3.3). Synthetic points, $k = 1$.

<!-- slide 119 -->
## Example: Irrelevant Features in Two Dimensions

<img src="../assets/week9/irrelevant-v0.svg" class="figure figure-plain" style="max-height:440px" alt="Twelve points on one line: circles on the left, triangles on the right. A test point just right of center has neighbors triangle, triangle, circle, so it is predicted triangle.">

<img src="../assets/week9/irrelevant-v1.svg" class="figure figure-plain" style="max-height:440px" alt="The same points spread out vertically by a random second feature. The test point's three nearest neighbors are now all circles, so it is predicted circle.">

"every feature is used, and they are all used the same amount. This means that if you have data with only a few relevant features and lots of irrelevant features, KNN is likely to do poorly." (CIML §3.2). Synthetic points.

<!-- slide 120 -->
## Example: Same Error, Different Entropy

<img src="../assets/week9/majority-tie-v1.svg" class="figure figure-plain" style="max-height:420px" alt="Two candidate splits of the same 1,200 examples, 800 yes and 400 no. Feature 1 makes two children of 400 yes, 200 no each. Feature 2 makes children of 250 yes, 240 no and 550 yes, 160 no. Both have error 400 out of 1200; weighted entropy is 0.92 for Feature 1 and 0.86 for Feature 2.">

Source: Elena Zheleva, CS 418 lecture slides, Feb. 19, 2026 (based on Marine Carpuat). "Both have the same error rate!!! Which is 'progressing more' towards a lower error?"

<!-- slide 121 -->
## Example: Growing a Tree in Two Dimensions

<img src="../assets/week9/tree-grow-v1.svg" class="figure figure-plain" style="max-height:430px" alt="Depth 1: one vertical cut splits the plane; the tree beside it has a root question on x1 and two leaves. Training accuracy rises to about 82 percent.">

<img src="../assets/week9/tree-grow-v3.svg" class="figure figure-plain" style="max-height:430px" alt="Depth 3: more cuts make smaller rectangles; the tree has eight leaves, about 92 percent training accuracy.">

<img src="../assets/week9/tree-grow-v4.svg" class="figure figure-plain" style="max-height:430px" alt="The full tree: many thin rectangles, some wrapped around single points, 100 percent training accuracy; too many leaves to draw.">

Every split is an axis-aligned cut. Synthetic; three labels flipped on purpose.

<!-- slide 122 -->
# Appendix: Minkowski Distance

<!-- slide 123 -->
## Minkowski Distance

A family of distances for continuous attributes, with $p$ attributes:

- **Minkowski**: $d(a, b) = \left( \sum_{i=1}^{p} |a_i - b_i|^r \right)^{1/r}$, "where $r$ is a parameter"
    - $r = 1$ gives Manhattan; $r = 2$ gives Euclidean
    - (or [`scipy.spatial.distance.minkowski`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.minkowski.html), with `p` = $r$ 🙂)

<!-- slide 124 -->
# Appendix: Pruning

<!-- slide 125 -->
## Pruning

"In some cases, we may want to stop the decision tree induction algorithm early to avoid overfitting." (Bello)

- **Pre-pruning (early stopping)**: "the decision tree induction algorithm is stopped before generating the full tree (for example, when the observed improvement in the estimate of the generalization error is below a certain threshold)." (Bello) Max depth is one such rule.
- **Post-pruning**: "the decision tree is initially grown to its maximum size and is then trimmed. The tree pruning step terminates when no further improvement in the estimate of the generalization error is observed." (Bello)
- Pruning has "the advantage that different branches can have different depths, accounting for the fact that the amount of data that gets passed down each branch might differ dramatically." (CIML)

<!-- slide 126 -->
# Sources

1. GitHub source: <https://github.com/jackbandy/data-science-fun/blob/main/docs/slides/week9.qmd>.
1. Last modified and compiled October 21, 12:12h Central (Chicago) time.
2. Slides developed using materials from [Elena Zheleva](https://www.cs.uic.edu/~elena/) and [Gonzalo Bello Lander](https://cs.uic.edu/profiles/gonzalo-bello/), the Berkeley DS 100 team, Marine Carpuat, and Brian Ziebart.
3. Slide deck built with [Quarto](https://quarto.org/) revealjs.
4. Title font is Big Shoulders; Body font is [Libre Franklin](https://en.wikipedia.org/wiki/Franklin_Gothic#Libre_Franklin).
5. Hal Daumé III, *[A Course in Machine Learning](http://ciml.info/)*, v0.99, [ch. 1](http://ciml.info/dl/v0_99/ciml-v0_99-ch01.pdf) ("Decision Trees"), [ch. 2](http://ciml.info/dl/v0_99/ciml-v0_99-ch02.pdf) ("Limits of Learning"), [ch. 3](http://ciml.info/dl/v0_99/ciml-v0_99-ch03.pdf) ("Geometry and Nearest Neighbors"), [ch. 7](http://ciml.info/dl/v0_99/ciml-v0_99-ch07.pdf) ("Linear Models"), and [ch. 11](http://ciml.info/dl/v0_99/ciml-v0_99-ch11.pdf) ("Kernel Methods").
6. Gonzalo Bello Lander, CS 418 Lecture 12 ("Classification (I): Basic Concepts & Decision Trees") and Lecture 13 ("Classification (II): More Classification Techniques"), 2026. The tax-cheating tree and Exercises 12.2–12.4 and 13.1 are from these slides.
7. Elena Zheleva, CS 418 lecture slides, Feb. 19, 24, and 26 and Mar. 3, 2026 (using materials by Marine Carpuat, Lise Getoor, and Brian Ziebart).
8. Jake VanderPlas, *[Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/)*, ["In-Depth: Decision Trees and Random Forests"](https://jakevdp.github.io/PythonDataScienceHandbook/05.08-random-forests.html) and ["In-Depth: Support Vector Machines"](https://jakevdp.github.io/PythonDataScienceHandbook/05.07-support-vector-machines.html), text CC BY-NC-ND.
9. Stephanie Yee and Tony Chu, ["A Visual Introduction to Machine Learning, Part 1"](https://r2d3.us/visual-intro-to-machine-learning-part-1/), R2D3.
10. Sebastian Raschka, ["Why are we growing decision trees via entropy instead of the classification error?"](https://sebastianraschka.com/faq/docs/decisiontree-error-vs-entropy.html), Machine Learning FAQ.
11. scikit-learn developers, [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) documentation, v1.8.
12. Week 9 figures: [`make_week9_figures.py`](https://github.com/jackbandy/data-science-fun/blob/main/docs/assets/week9/make_week9_figures.py); the class survey, birth months, point clouds, and study-hours students (Week 8's, plus a random day of month born), and practice-exam and seat splits are synthetic. The survey uses the same generator as the code demo. The Day 2 margin, $C$, kernel, and $\gamma$ figures use the Palmer Penguins; the one-feature lift and the five kernel shapes are synthetic; the movie-review paths come from the Python Yard demo's tree.
13. scikit-learn developers, [`KNeighborsClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) and [`plot_tree`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) documentation.
14. scikit-learn developers, ["RBF SVM parameters"](https://scikit-learn.org/stable/auto_examples/svm/plot_rbf_parameters.html) and user guide §1.4, ["Support Vector Machines"](https://scikit-learn.org/stable/modules/svm.html).
15. Tiffany Timbers, Trevor Campbell, Melissa Lee, Joel Ostblom, and Lindsey Heagy, *[Data Science: A First Introduction (Python Edition)](https://python.datasciencebook.ca/)*, ch. 5 and 6.
16. Movie reviews: Hou, Li, He, Yan, Chen, and McAuley, [Amazon Reviews 2023](https://amazon-reviews-2023.github.io/), `Movies_and_TV`; course sample in [`datasets/movie-reviews`](https://github.com/jackbandy/data-science-fun/tree/main/datasets/movie-reviews).
17. Penguins: Allison Horst, Alison Hill, and Kristen Gorman, [palmerpenguins](https://doi.org/10.5281/zenodo.3960218) (2020), CC0; data originally from Kristen B. Gorman, Tony D. Williams, and William R. Fraser, "Ecological Sexual Dimorphism and Environmental Variability within a Community of Antarctic Penguins (Genus *Pygoscelis*)," *PLoS ONE* 9(3): e90081 (2014), <https://doi.org/10.1371/journal.pone.0090081>.